# 01a — Extract Elevation (OpenTopoData)
**Data source:** [OpenTopoData API](https://www.opentopodata.org/) — ASTER 30m DEM

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00 output

**Output:** `elevation.parquet` (one row per unique station)

**Estimated time:** ~10-20 min for ~170 stations (rate limited: 1 req/sec)

> Enable Internet in Kaggle settings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, time, requests, logging
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# === Logging Setup ===
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01a_elevation')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

# === Paths ===
INPUT_DIR  = '/kaggle/input/ey-water-quality-nb00'  # output dataset from notebook 00
OUTPUT_DIR = '/kaggle/working'

# Column config (must match notebook 00)
LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'

log.info(f'Input: {INPUT_DIR}')
log.info(f'Output: {OUTPUT_DIR}')

In [ ]:
# Load base data
train_base = pd.read_parquet(f'{INPUT_DIR}/train_base.parquet')
val_base   = pd.read_parquet(f'{INPUT_DIR}/val_base.parquet')

all_data = pd.concat([train_base, val_base], ignore_index=True)
unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()

log.info(f'Train: {train_base.shape}, Val: {val_base.shape}')
log.info(f'Unique stations to extract: {len(unique_stations)}')

---
## Extraction

In [ ]:
def fetch_elevation(lat, lon, retries=3):
    """Fetch elevation from OpenTopoData ASTER 30m DEM."""
    url = 'https://api.opentopodata.org/v1/aster30m'
    for attempt in range(retries):
        try:
            r = requests.get(url, params={'locations': f'{lat},{lon}'}, timeout=30)
            r.raise_for_status()
            return float(r.json()['results'][0]['elevation'])
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                return np.nan
    return np.nan

In [ ]:
total = len(unique_stations)
results = []
start_time = time.time()
failed = 0

log.info(f'Starting elevation extraction for {total} stations...')

for i, row in unique_stations.iterrows():
    lat, lon = row[LAT_COL], row[LON_COL]
    station = row[STATION_COL]
    
    elev = fetch_elevation(lat, lon)
    results.append({
        STATION_COL: station,
        LAT_COL: lat,
        LON_COL: lon,
        'elevation_m': elev
    })
    
    if np.isnan(elev):
        failed += 1
    
    # Progress logging every 10 stations
    done = len(results)
    if done % 10 == 0 or done == total:
        elapsed = time.time() - start_time
        rate = done / elapsed if elapsed > 0 else 0
        eta = (total - done) / rate if rate > 0 else 0
        log.info(f'  [{done:3d}/{total}] {done/total*100:5.1f}% | '
                 f'elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | '
                 f'failed {failed} | last: {station[:20]} elev={elev}')
    
    # Rate limit: OpenTopoData allows 1 req/sec
    time.sleep(1.0)

elapsed_total = time.time() - start_time
log.info(f'DONE in {elapsed_total/60:.1f} min | {total} stations | {failed} failed')

In [ ]:
# Build output dataframe
elev_df = pd.DataFrame(results)

log.info(f'Output shape: {elev_df.shape}')
log.info(f'Nulls: {elev_df["elevation_m"].isnull().sum()}')
log.info(f'Elevation range: {elev_df["elevation_m"].min():.0f} – {elev_df["elevation_m"].max():.0f} m')

display(elev_df.describe())

---
## Figure: Elevation Map

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))

sc = ax.scatter(elev_df[LON_COL], elev_df[LAT_COL],
                c=elev_df['elevation_m'], cmap='terrain', s=70,
                edgecolors='gray', linewidths=0.3, alpha=0.9)
cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Elevation (m)', fontsize=11)

# Mark failed stations
failed_mask = elev_df['elevation_m'].isnull()
if failed_mask.any():
    ax.scatter(elev_df.loc[failed_mask, LON_COL], elev_df.loc[failed_mask, LAT_COL],
              c='red', marker='x', s=100, linewidths=2, label='Failed', zorder=5)
    ax.legend(fontsize=10)

ax.set_xlim(16, 33); ax.set_ylim(-35, -22)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'Elevation per Station\n{(~failed_mask).sum()}/{len(elev_df)} stations extracted')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_01a_elevation_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Save Output

In [ ]:
out_path = f'{OUTPUT_DIR}/elevation.parquet'
elev_df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024

log.info(f'Saved: {out_path} ({size_kb:.1f} KB, {len(elev_df)} rows)')
log.info(f'Columns: {elev_df.columns.tolist()}')
print(f'\n=== DONE ===')
print(f'Output: elevation.parquet')
print(f'Rows: {len(elev_df)}')
print(f'Next: add this notebook output as dataset input for 01b/01e')